# Week 13 과제 · matplotlib 데이터 시각화

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MNU-AI-Programming/AI-Assisted-Python-Programming/blob/main/weeks/week13/assignment.ipynb)

## 제출 방법

1. **파일 → 드라이브에 사본 저장** 후 파일 이름을 `week13_학번_이름.ipynb`로 바꿉니다.
2. `# TODO` 부분을 채우고 각 문제 아래 **자동 확인 셀**이 ✅인지 확인합니다. 자동 확인은 그래프 객체(`ax`)의 제목·레이블·막대·점을 코드로 검사합니다.
3. 각 문제 마지막 셀의 그래프를 **눈으로도 확인**하고, AI 활용 기록과 성찰을 작성합니다.
4. **런타임 → 다시 시작 및 모두 실행** 후 `.ipynb` 파일을 제출합니다.

> **규칙**: 모든 그래프 함수는 `fig, ax = plt.subplots(...)`로 만들고 **`(fig, ax)` 또는 `fig`를 돌려줍니다.** 함수 안에서 `plt.show()`는 부르지 않습니다.

| 문제 | 내용 | 배점 |
|---|---|---:|
| 1 | 강조가 있는 메뉴 매출 가로 막대 | 2 |
| 2 | 일 매출 추세와 이동평균 | 2 |
| 3 | 기온과 아이스 비율 산점도 + 추세선 | 3 |
| 4 | 2×2 대시보드 저장 (AI 협업) | 3 |
| | **합계** | **10** |

## 준비: 데이터와 자동 확인 도우미

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import tempfile
from pathlib import Path
import urllib.request

try:
    import koreanize_matplotlib
except ModuleNotFoundError:
    %pip install -q koreanize-matplotlib
    import koreanize_matplotlib

DATA_URL = "https://raw.githubusercontent.com/MNU-AI-Programming/AI-Assisted-Python-Programming/main/datasets/"

def get_data(name):
    for folder in [Path("../../datasets"), Path("datasets"), Path(".")]:
        if (folder / name).exists():
            return str(folder / name)
    urllib.request.urlretrieve(DATA_URL + name, name)
    return name

CAFE = pd.read_csv(get_data("cafe_sales.csv"), parse_dates=["date"])
WEATHER = pd.read_csv(get_data("weather_daily.csv"), parse_dates=["date"])
CAFE["revenue"] = CAFE["unit_price"] * CAFE["qty"]
TMP = Path(tempfile.mkdtemp())


def check(desc, test):
    """자동 확인 도우미. 검사 중 만든 그림은 화면에 남기지 않고 닫는다."""
    try:
        ok = test()
    except NotImplementedError:
        print(f"⏳ 미작성 - {desc}")
    except Exception as e:
        print(f"❌ 오류 - {desc} ({type(e).__name__}: {e})")
    else:
        print(f"{'✅ 통과' if ok else '❌ 실패'} - {desc}")
    finally:
        plt.close("all")


def title_of(ax):
    return " ".join(ax.get_title(loc=l) for l in ("left", "center", "right")).strip()

---
## 문제 1. 강조가 있는 메뉴 매출 가로 막대 (2점)

`plot_menu_share(cafe)`는 메뉴별 **매출 비중(%)** 가로 막대 그래프를 그리고 `(fig, ax)`를 돌려줍니다.

- 막대는 **위에서 아래로 비중이 큰 순서** (가장 큰 메뉴가 맨 위)
- **가장 큰 메뉴 하나만 다른 색**, 나머지는 모두 같은 색
- 제목: 결론을 담은 문장 (10글자 이상), x축 레이블에 `%` 포함
- 각 막대 끝에 값 표시 (`ax.bar_label`)

> 힌트: `barh`는 첫 번째 막대를 **아래**에 그립니다. 오름차순으로 정렬해 그리거나 `ax.invert_yaxis()`를 쓰세요.

In [ ]:
def plot_menu_share(cafe):
    # TODO: 여기에 코드를 작성하세요
    raise NotImplementedError("코드를 작성하세요")

In [ ]:
def _bars_top_to_bottom(ax):
    """화면 위에서 아래 순서의 (메뉴, 길이, 색)."""
    ax.figure.canvas.draw()
    items = sorted(ax.patches, key=lambda p: -(p.get_y() if not ax.yaxis_inverted() else -p.get_y()))
    labels = {round(t.get_position()[1], 3): t.get_text() for t in ax.get_yticklabels()}
    return [(labels.get(round(p.get_y() + p.get_height() / 2, 3)), p.get_width(), mcolors.to_hex(p.get_facecolor())) for p in items]

check("막대 11개", lambda: len(plot_menu_share(CAFE)[1].patches) == 11)
check("맨 위가 가장 큰 메뉴(아메리카노)", lambda: _bars_top_to_bottom(plot_menu_share(CAFE)[1])[0][0] == "아메리카노")
check("위에서 아래로 내림차순", lambda: [w for _, w, _ in _bars_top_to_bottom(plot_menu_share(CAFE)[1])]
      == sorted([w for _, w, _ in _bars_top_to_bottom(plot_menu_share(CAFE)[1])], reverse=True))
check("비중 합 100", lambda: abs(sum(p.get_width() for p in plot_menu_share(CAFE)[1].patches) - 100) < 1e-6)
check("색은 2가지, 강조는 1개", lambda: (lambda cs: len(set(cs)) == 2 and min(cs.count(c) for c in set(cs)) == 1)(
      [c for _, _, c in _bars_top_to_bottom(plot_menu_share(CAFE)[1])]))
check("제목 10글자 이상, x축 %", lambda: len(title_of(plot_menu_share(CAFE)[1])) >= 10 and "%" in plot_menu_share(CAFE)[1].get_xlabel())
check("값 표시(bar_label)", lambda: len(plot_menu_share(CAFE)[1].texts) >= 11)

In [ ]:
fig, ax = plot_menu_share(CAFE)
plt.show()

---
## 문제 2. 일 매출 추세와 이동평균 (2점)

`plot_trend(cafe, window=7)`은 **일 매출(천 원)** 과 **`window`일 이동평균(천 원)** 을 선 두 개로 그리고 `(fig, ax)`를 돌려줍니다.

- 첫 번째 선: 일 매출 (연한 색, 얇게), 두 번째 선: 이동평균 (진한 색, 두껍게)
- x축은 날짜, 범례에 `"일 매출"`, `f"{window}일 이동평균"` 표시
- y축 레이블에 단위 `천 원` 포함, 제목 있음
- 이동평균은 `rolling(window).mean()` 사용 (앞부분 NaN은 그대로 둠)

In [ ]:
def plot_trend(cafe, window=7):
    # TODO: 여기에 코드를 작성하세요
    raise NotImplementedError("코드를 작성하세요")

In [ ]:
_daily = CAFE.groupby("date")["revenue"].sum() / 1000
check("선 2개", lambda: len(plot_trend(CAFE)[1].lines) == 2)
check("첫 선 = 일 매출 값", lambda: np.allclose(plot_trend(CAFE)[1].lines[0].get_ydata(), _daily.values))
check("둘째 선 = 7일 이동평균", lambda: np.allclose(plot_trend(CAFE)[1].lines[1].get_ydata(), _daily.rolling(7).mean().values, equal_nan=True))
check("window 매개변수", lambda: np.allclose(plot_trend(CAFE, 3)[1].lines[1].get_ydata(), _daily.rolling(3).mean().values, equal_nan=True))
check("범례", lambda: [t.get_text() for t in plot_trend(CAFE, 5)[1].get_legend().get_texts()] == ["일 매출", "5일 이동평균"])
check("이동평균 선이 더 두꺼움", lambda: (lambda ax: ax.lines[1].get_linewidth() > ax.lines[0].get_linewidth())(plot_trend(CAFE)[1]))
check("y축 단위, 제목", lambda: "천 원" in plot_trend(CAFE)[1].get_ylabel() and title_of(plot_trend(CAFE)[1]) != "")

In [ ]:
fig, ax = plot_trend(CAFE)
plt.show()

---
## 문제 3. 기온과 아이스 비율 산점도 + 추세선 (3점)

`plot_temp_ice(cafe, weather)`는 영업일마다 **평균기온(x)** 과 **아이스 음료 비율 %(y)** 를 산점도로 그리고, `np.polyfit(x, y, 1)`로 구한 **추세선**을 겹쳐 그립니다. `(fig, ax, slope)`를 돌려주며 `slope`는 추세선 기울기(소수 둘째 자리)입니다.

- 아이스 비율 = ICE 수량 ÷ (ICE + HOT 수량) × 100 (`temp_type`이 `"-"`인 디저트 제외)
- 점의 개수 = 영업일 수 (95)
- 축 레이블에 단위(`℃`, `%`), 제목에 **상관은 인과가 아님**을 해치지 않는 표현 사용 (예: "~할수록 ~이 높다")

In [ ]:
def plot_temp_ice(cafe, weather):
    # TODO: 여기에 코드를 작성하세요
    raise NotImplementedError("코드를 작성하세요")

In [ ]:
EXPECTED_SLOPE = 2.9           # cafe_sales.csv + weather_daily.csv 기준값
check("점 95개", lambda: len(plot_temp_ice(CAFE, WEATHER)[1].collections[0].get_offsets()) == 95)
check("추세선 1개", lambda: len(plot_temp_ice(CAFE, WEATHER)[1].lines) == 1)
check("기울기", lambda: plot_temp_ice(CAFE, WEATHER)[2] == EXPECTED_SLOPE)
check("y값은 0~100 (%)", lambda: (lambda o: o[:, 1].min() >= 0 and o[:, 1].max() <= 100 and o[:, 1].max() > 1)(
      np.asarray(plot_temp_ice(CAFE, WEATHER)[1].collections[0].get_offsets())))
check("축 단위", lambda: "℃" in plot_temp_ice(CAFE, WEATHER)[1].get_xlabel() and "%" in plot_temp_ice(CAFE, WEATHER)[1].get_ylabel())
check("제목 있음", lambda: title_of(plot_temp_ice(CAFE, WEATHER)[1]) != "")

In [ ]:
fig, ax, slope = plot_temp_ice(CAFE, WEATHER)
plt.show()
print(f"기온 1℃당 아이스 비율 약 {slope}%p")

---
## 문제 4. 2×2 대시보드 저장 (AI 협업, 3점)

`make_dashboard(cafe, weather, path)`는 2×2 대시보드를 만들어 `path`에 **PNG(dpi 150 이상)** 로 저장하고 `fig`를 돌려줍니다.

| 위치 | 내용 |
|---|---|
| 왼쪽 위 | 일 매출 + 7일 이동평균 (선 2개) |
| 오른쪽 위 | 메뉴별 매출 비중 가로 막대 |
| 왼쪽 아래 | **요일별 일평균 매출** 세로 막대 (월~토 순서, **y축 0부터**) |
| 오른쪽 아래 | 기온 vs 아이스 비율 산점도 |

- 네 칸 모두 제목이 있고, 그림 전체 제목(`fig.suptitle`)도 있어야 합니다.
- 문제 1~3의 코드를 재사용해도 됩니다. (함수를 `ax`를 받도록 바꾸는 리팩토링을 해 보세요.)

**AI 협업 과제**: AI에게 대시보드 코드를 요청한 뒤, 실습에서 만든 **체크리스트**로 AI의 첫 결과를 점검하고 고친 점을 기록하세요.

In [ ]:
def make_dashboard(cafe, weather, path):
    # TODO: 여기에 코드를 작성하세요
    raise NotImplementedError("코드를 작성하세요")

In [ ]:
dash_path = TMP / "dashboard.png"


def _dash():
    return make_dashboard(CAFE, WEATHER, dash_path)

check("Axes 4개", lambda: len([a for a in _dash().axes if a.get_label() != "<colorbar>"]) >= 4)
check("모든 칸에 제목", lambda: all(title_of(a) for a in _dash().axes[:4]))
check("전체 제목(suptitle)", lambda: _dash()._suptitle is not None and _dash()._suptitle.get_text() != "")
check("왼쪽 위: 선 2개", lambda: len(_dash().axes[0].lines) == 2)
check("왼쪽 아래: 막대 6개, y축 0부터", lambda: (lambda ax: len(ax.patches) == 6 and ax.get_ylim()[0] == 0)(_dash().axes[2]))
check("오른쪽 아래: 점 95개", lambda: len(_dash().axes[3].collections[0].get_offsets()) == 95)
check("PNG 저장 (dpi 150 이상)", lambda: (_dash(), dash_path.exists() and dash_path.stat().st_size > 50_000)[1])

In [ ]:
try:
    fig = make_dashboard(CAFE, WEATHER, TMP / "dashboard.png")
    plt.show()
except NotImplementedError:
    print("아직 작성하지 않았습니다.")

### AI 활용 기록 (문제 4)

| 체크리스트 항목 | AI 첫 결과 | 내가 고친 내용 |
|---|---|---|
| 질문에 맞는 차트 종류 |  |  |
| 제목·축 단위 |  |  |
| 막대 0부터, 의미 있는 순서 |  |  |
| 색의 의미 |  |  |
| 데이터 값과 그래프 일치 |  |  |

사용한 AI 도구와 프롬프트:

---
## 성찰 (필수)

1. "제목에 결론을 쓴다"는 원칙을 적용하면서 어떤 점이 어려웠나요?
2. 그래프를 코드로 검사(자동 확인)하는 방법이 눈으로 보는 것과 비교해 어떤 장단점이 있나요?
3. 최종 프로젝트에서 쓸 그래프 2~3개를 이번 주 원칙에 따라 계획해 보세요.

(여기에 작성)